# Unidade 3 — Guia de estudo

[▶ Abrir este notebook no Google Colab](https://colab.research.google.com/github/lalvim/disciplina_computacao_aplicada_humanidades_digitais/blob/main/unidade_03/00_guia_da_unidade.ipynb)

## Como transformar fontes em dados analisáveis?

**Problema orientador:** como preparar documentos e registros produzidos em
formatos diferentes sem perder a ligação com as fontes e decisões?

O protocolo da Unidade 2 definiu o que coletar. Agora construiremos uma
primeira base processável. “Limpar” não significa tornar os dados neutros:
significa aplicar regras explícitas, preservar originais, testar resultados e
documentar perdas. Rawson e Muñoz (2019) alertam que a ideia de limpeza pode
ocultar trabalho interpretativo e relações de autoridade.

![Fontes em papel, imagem e tabela atravessam camadas transparentes de transformação e produzem tabelas, relações e segmentos de texto sempre ligados aos originais preservados.](https://github.com/lalvim/disciplina_computacao_aplicada_humanidades_digitais/blob/main/unidade_03/imagens/00_abertura_conceitual.png?raw=1)

A ilustração antecipa o argumento central: transformar não rompe o vínculo
com a fonte. Cada produto derivado precisa manter um caminho de retorno às
entradas, regras e decisões que o produziram.

In [ ]:
# @title Preparação do ambiente — execute esta célula no Google Colab
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

URL_REPOSITORIO = 'https://github.com/lalvim/disciplina_computacao_aplicada_humanidades_digitais.git'
REPOSITORIO = Path(
    "/content/disciplina_computacao_aplicada_humanidades_digitais"
)
PASTA_UNIDADE = REPOSITORIO / 'unidade_03'

try:
    import google.colab  # type: ignore  # noqa: F401
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

if EM_COLAB:
    if not (REPOSITORIO / ".git").exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "main",
                URL_REPOSITORIO,
                str(REPOSITORIO),
            ],
            check=True,
        )

    PACOTES_COLAB = [('openpyxl', 'openpyxl>=3.1,<4'), ('pypdf', 'pypdf>=5,<6'), ('PIL', 'Pillow>=11,<12')]
    ausentes = [
        especificacao
        for modulo, especificacao in PACOTES_COLAB
        if importlib.util.find_spec(modulo) is None
    ]
    if ausentes:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", *ausentes],
            check=True,
        )

    os.chdir(PASTA_UNIDADE)
    print("Ambiente preparado em:", Path.cwd())
else:
    print("Ambiente local: nenhuma clonagem necessária.")

Ambiente preparado em: /content/disciplina_computacao_aplicada_humanidades_digitais/unidade_03


O problema orientador define a exigência de preservar vínculos durante a
transformação. Para realizá-la, precisamos decompor o trabalho em etapas nas
quais cada produto possa ser verificado antes de alimentar o seguinte.

## Objetivos e percurso

| Notebook | Questão | Produto |
|---|---|---|
| 00 | Como trabalhar sem perder rastros? | Diagnóstico |
| 01 | Como ler formatos e extrair texto? | Inventário técnico |
| 02 | Como estruturar e corrigir com cautela? | Tabela intermediária |
| 03 | Como juntar e integrar sem criar casos? | Base integrada |
| 04 | A base está processável e auditável? | Primeira versão da base |

A unidade usa dados fictícios deliberadamente inconsistentes e um pequeno
extrato didático dos códigos de municípios do IBGE, acompanhado de
proveniência. Não há coleta pela internet durante a execução.

![Quatro etapas ligam leitura de formatos, estruturação, integração e empacotamento; os dados brutos permanecem preservados e uma seta de retorno representa auditoria e revisão.](https://github.com/lalvim/disciplina_computacao_aplicada_humanidades_digitais/blob/main/unidade_03/imagens/00_percurso_unidade.svg?raw=1)

O percurso é cumulativo: cada notebook produz uma representação que será
retomada pelo seguinte. Testes e limites podem exigir retorno a qualquer
decisão anterior.

In [ ]:
from pathlib import Path
import pandas as pd

raiz_dados = Path("dados")
bruto = pd.read_csv(raiz_dados / "brutos" / "catalogo_messy.csv", sep=";")
print("Registros brutos:", len(bruto))
print("Arquivos brutos:", len(list((raiz_dados / "brutos").iterdir())))
bruto.head(3)

Registros brutos: 8
Arquivos brutos: 15


,id_documento,titulo,data_documento,municipio,codigo_municipio,genero,palavras
0,D001,Jornal Aurora,1890-01-05,São Paulo,3550308,Editorial,860.0
1,D002,gazeta popular,06/02/1891,Sao paulo,3550308,notícia,610.0
2,D003,Correio do Vale,1892,Belo Horizonte,3106200,Carta,NaN


A inspeção apresentou os materiais que atravessarão o percurso. Antes de
transformá-los, precisamos estabelecer onde cada estado dos dados ficará e
como será possível retornar ao ponto de entrada.

## Regra de ouro: camadas, não sobrescrita

- `brutos/`: cópias recebidas, imutáveis;
- `intermediarios/`: resultados de etapas verificáveis;
- `derivados/`: produtos destinados à análise;
- notebooks: ordem e justificativa das operações.

Uma cópia bruta não é “verdade original”; ela é o ponto de entrada preservado
do fluxo. Proveniência continua necessária.

Separar as camadas protege os dados recebidos, mas não identifica quais
operações serão necessárias em cada projeto. O diagnóstico antecipa essas
operações e os riscos que deverão ser registrados ao longo da unidade.

## Diagnóstico

Que transformações seu projeto exigirá? Quais podem alterar sentido, unidade
ou quantidade de registros?

**Resposta:**

O projeto exigirá a conversão dos 12 discursos em .txt em uma tabela de parágrafos (trecho_id) via regex e Pandas, essa transformação altera a unidade (do texto integral para o parágrafo) e a quantidade de registros (expandindo de 12 documentos brutos para 60–120 trechos derivados), enquanto o isolamento do parágrafo pode alterar pontualmente o sentido semântico se desconectado do discurso completo.

## Produto e avaliação

A entrega incluirá inventário, código executável, tabela processável, log de
transformações, testes de chaves/junções, amostra de controle de OCR e limites.
Frequências substantivas, gráficos e hipóteses exploratórias pertencem à
Unidade 4. Consulte `referencias.md` para leituras completas. Comece pelo
Notebook 01, no qual os formatos recebidos determinarão leitores, parâmetros
e estratégias de extração diferentes.